In [ ]:
!pip install -q executorch
import executorch, torch, torchvision, os
from importlib.metadata import version
print("executorch:", version("executorch"))
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pth", ".json")):
            print(os.path.join(root, f))

In [ ]:
!pip install -q torchvision "torch==2.15.0"
import executorch, torch, torchvision, os
from importlib.metadata import version
print("executorch:", version("executorch"))
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".pth", ".json")):
            print(os.path.join(root, f))

In [ ]:
import json, torch, torch.nn as nn
from torchvision import models
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower
from executorch.runtime import Runtime

D = "/kaggle/input/datasets/viroshans/crop-model-v3"
CL = json.load(open(f"{D}/classes_v3.json"))
m = models.mobilenet_v3_small()
m.classifier[3] = nn.Linear(m.classifier[3].in_features, len(CL))
m.load_state_dict(torch.load(f"{D}/best_model_v3.pth", map_location="cpu"))
m.eval()

x = torch.randn(1, 3, 224, 224)
ep = torch.export.export(m, (x,))
et = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()]).to_executorch()
with open("/kaggle/working/vision.pte", "wb") as f:
    f.write(et.buffer)
print("saved vision.pte, size MB:", round(len(et.buffer) / 1e6, 2), "| classes:", len(CL))

# verify: the .pte output must match the PyTorch model
prog = Runtime.get().load_program("/kaggle/working/vision.pte")
out = prog.load_method("forward").execute([x])[0]
with torch.no_grad():
    ref = m(x)
print("max abs difference:", (out - ref).abs().max().item())
print("same top class:", out.argmax().item() == ref.argmax().item())

In [ ]:
!pip install -q "executorch==1.5.1"
import torch, executorch
print("torch:", torch.__version__)
print("executorch:", executorch.__version__)

In [ ]:
from importlib.metadata import version
print("executorch:", version("executorch"))
print("torch:", version("torch"))
try:
    import torchvision
    from torchvision import models
    models.mobilenet_v3_small(weights=None)
    print("torchvision:", torchvision.__version__, "- works")
except Exception as e:
    print("torchvision problem:", repr(e)[:300])

In [ ]:
import glob, os, torch, torch.nn as nn
from torchvision import models
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower
from executorch.runtime import Runtime

pth = glob.glob("/kaggle/input/**/best_model_v3.pth", recursive=True)[0]
print("Using:", pth)

model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 13)
model.load_state_dict(torch.load(pth, map_location="cpu"))
model.eval()

example = torch.randn(1, 3, 224, 224)
ep = torch.export.export(model, (example,))
et = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()]).to_executorch()
with open("/kaggle/working/vision.pte", "wb") as f:
    f.write(et.buffer)
print("vision.pte size (MB):", round(os.path.getsize("/kaggle/working/vision.pte") / 1e6, 2))

torch.manual_seed(0)
x = torch.randn(1, 3, 224, 224)
method = Runtime.get().load_program("/kaggle/working/vision.pte").load_method("forward")
out_pte = method.execute([x])[0]
out_ref = model(x)
print("max abs difference:", (out_pte - out_ref).abs().max().item())
print("same top class:", out_pte.argmax().item() == out_ref.argmax().item())

In [ ]:
import torch, torch.nn as nn
from torchvision import models
import glob

pth = glob.glob("/kaggle/input/**/best_model_v3.pth", recursive=True)[0]
model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 13)
model.load_state_dict(torch.load(pth, map_location="cpu"))
model.eval()

torch.manual_seed(0)
x = torch.randn(1, 3, 224, 224).contiguous()
ref = model(x)

out = None
try:
    from executorch.extension.pybindings.portable_lib import _load_for_executorch
    m = _load_for_executorch("/kaggle/working/vision.pte")
    out = m.forward([x])[0]
    print("checked with portable_lib")
except Exception as e:
    print("portable_lib failed:", repr(e)[:200])
    from executorch.runtime import Runtime
    method = Runtime.get().load_program("/kaggle/working/vision.pte").load_method("forward")
    out = method.execute([x.clone().detach()])[0]
    print("checked with Runtime")

print("max abs difference:", (out - ref).abs().max().item())
print("same top class:", out.argmax().item() == ref.argmax().item())

In [1]:
import torch, torch.nn as nn, glob
from torchvision import models
from executorch.runtime import Runtime

pth = glob.glob("/kaggle/input/**/best_model_v3.pth", recursive=True)[0]
model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 13)
model.load_state_dict(torch.load(pth, map_location="cpu"))
model.eval()

torch.manual_seed(0)
x = torch.randn(1, 3, 224, 224)
ref = model(x)
method = Runtime.get().load_program("/kaggle/working/vision.pte").load_method("forward")
out = method.execute([x])[0]
print("max abs difference:", (out - ref).abs().max().item())
print("same top class:", out.argmax().item() == ref.argmax().item())

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu0/regs/identification/midr_el1
[cpuinfo_utils.cpp:109] Failed to open midr file /sys/devices/system/cpu/cpu0/regs/identification/midr_el1
[cpuinfo_utils.cpp:125] CPU info and manual query on # of cpus dont match.


RuntimeError: tensor does not have a device
Exception raised from device_default at /__w/_temp/conda_environment_35382918402/lib/python3.13/site-packages/torch/include/c10/core/TensorImpl.h:1303 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x9d (0x7c76ddb843fd in /usr/local/lib/python3.13/dist-packages/torch/lib/libc10.so)
frame #1: c10::detail::torchCheckFail(char const*, char const*, unsigned int, char const*) + 0x69 (0x7c76ddb18a43 in /usr/local/lib/python3.13/dist-packages/torch/lib/libc10.so)
frame #2: <unknown function> + 0x3bb57 (0x7c76c2786b57 in /usr/local/lib/python3.13/dist-packages/executorch/extension/pybindings/_C.cpython-313-x86_64-linux-gnu.so)
frame #3: <unknown function> + 0x3c642 (0x7c76c2787642 in /usr/local/lib/python3.13/dist-packages/executorch/extension/pybindings/_C.cpython-313-x86_64-linux-gnu.so)
frame #4: <unknown function> + 0x5432d (0x7c76c279f3